# 編集距離と n-gram 言語モデルによる誤り検出・訂正

書籍 第3章「系列ラベリング問題の解き方」（3-3 誤り検出：誤り箇所の検出と訂正、p.104〜110）に対応する演習ノートブックです。
CPU のみ・数秒で実行でき、GECToR などの深層学習手法の前段階となる古典的・基礎的な仕組みを実装して確認します。

1. **編集距離（Levenshtein 距離）と隣接文字入れ替え（Damerau-Levenshtein 距離）**（3-3-1、脚注 42〜43）
2. **編集距離を用いたスペル訂正の候補ランキング**
3. **文字・単語 n-gram 言語モデルによる誤り検出**（3-3-3）

## 1. 編集距離（Levenshtein 距離）の実装（3-3-1）

2つの文字列 $A$（長さ $m$）と $B$（長さ $n$）の間で、「挿入・削除・置換」の最小操作回数を動的計画法（計算量 $O(mn)$）で求めます。
また、書籍 3-3-1（脚注 42〜43）で紹介されているように、キーボード入力では `the` を `hte` のように隣り合う 2 文字を入れ替える（swap）ミスがよく起きるため、入れ替えもコスト 1 と数える **Damerau-Levenshtein 距離** も実装して比較します。

In [1]:
import math
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

def levenshtein_dp(a, b):
    '''Levenshtein 距離と DP テーブルを返す（計算量 O(mn)）'''
    m, n = len(a), len(b)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    for i in range(m + 1):
        dp[i, 0] = i
    for j in range(n + 1):
        dp[0, j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i, j] = min(
                dp[i - 1, j] + 1,       # 削除
                dp[i, j - 1] + 1,       # 挿入
                dp[i - 1, j - 1] + cost # 置換（一致なら 0）
            )
    return int(dp[m, n]), dp

def damerau_levenshtein(a, b):
    '''隣接文字の入れ替え（swap）もコスト 1 とする編集距離'''
    m, n = len(a), len(b)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    for i in range(m + 1):
        dp[i, 0] = i
    for j in range(n + 1):
        dp[0, j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i, j] = min(dp[i - 1, j] + 1, dp[i, j - 1] + 1, dp[i - 1, j - 1] + cost)
            if i > 1 and j > 1 and a[i - 1] == b[j - 2] and a[i - 2] == b[j - 1]:
                dp[i, j] = min(dp[i, j], dp[i - 2, j - 2] + 1)  # swap
    return int(dp[m, n])

dist, dp_table = levenshtein_dp("hte", "the")
print(f"Levenshtein('hte', 'the')         = {dist}")
print(f"Damerau-Levenshtein('hte', 'the') = {damerau_levenshtein('hte', 'the')}")

print("\nDP テーブル ('hte' -> 'the'):")
display(pd.DataFrame(dp_table, index=["#"] + list("hte"), columns=["#"] + list("the")))

Levenshtein('hte', 'the')         = 2
Damerau-Levenshtein('hte', 'the') = 1

DP テーブル ('hte' -> 'the'):


,#,t,h,e
#,0,1,2,3
h,1,1,1,2
t,2,1,2,2
e,3,2,2,2


---

## 2. 編集距離を使ったスペル訂正の候補ランキング

辞書（単語リスト）の中から、入力語との編集距離が小さい順に訂正候補を並べます。

In [2]:
vocab = ["the", "they", "there", "their", "them", "then", "heat", "hat", "language", "natural", "processing", "model"]

typos = ["hte", "langage", "natrual", "proccessing"]
for w in typos:
    ranked = sorted(vocab, key=lambda v: (damerau_levenshtein(w, v), levenshtein_dp(w, v)[0], v))[:3]
    scores = [(cand, damerau_levenshtein(w, cand), levenshtein_dp(w, cand)[0]) for cand in ranked]
    print(f"入力: {w:12s} -> 候補 (単語, Damerau, Levenshtein): {scores}")

入力: hte          -> 候補 (単語, Damerau, Levenshtein): [('the', 1, 2), ('hat', 2, 2), ('them', 2, 3)]
入力: langage      -> 候補 (単語, Damerau, Levenshtein): [('language', 1, 1), ('hat', 6, 6), ('heat', 6, 6)]
入力: natrual      -> 候補 (単語, Damerau, Levenshtein): [('natural', 1, 2), ('hat', 5, 5), ('language', 5, 5)]
入力: proccessing  -> 候補 (単語, Damerau, Levenshtein): [('processing', 1, 1), ('model', 9, 9), ('their', 9, 9)]


---

## 3. n-gram 言語モデルによる誤り検出（3-3-3）

単語単位のスペルが正しくても、文脈に合わない誤用（例：`I like an apple` vs `I like a apple`、`自然言語処理` vs `自然言語処置`）は辞書引きだけでは検出できません。
ここでは Add-k スムージング付きの単語バイグラム言語モデルを構築し、文の**パープレキシティ（Perplexity; PPL）**（低いほど自然な文）を比較してみます。

In [3]:
train_sents = [
    "<s> i like an apple </s>",
    "<s> i eat an apple every day </s>",
    "<s> i like a banana </s>",
    "<s> she likes a book </s>",
    "<s> he reads a book every day </s>",
    "<s> we study natural language processing </s>",
    "<s> natural language processing is fun </s>",
] * 20

unigram_counts = Counter()
bigram_counts = defaultdict(Counter)
vocab_lm = set()

for s in train_sents:
    toks = s.split()
    vocab_lm.update(toks)
    for i, tok in enumerate(toks):
        unigram_counts[tok] += 1
        if i > 0:
            bigram_counts[toks[i - 1]][tok] += 1

def bigram_perplexity(sent, k=0.1):
    toks = ["<s>"] + sent.lower().split() + ["</s>"]
    V = len(vocab_lm)
    log_prob = 0.0
    n = len(toks) - 1
    for i in range(1, len(toks)):
        prev_w, curr_w = toks[i - 1], toks[i]
        num = bigram_counts[prev_w][curr_w] + k
        den = unigram_counts[prev_w] + k * V
        log_prob += math.log(num / den)
    return math.exp(-log_prob / n)

test_sents = [
    ("自然な文 1", "i like an apple"),
    ("誤りを含む文 1（冠詞ミス）", "i like a apple"),
    ("自然な文 2", "natural language processing is fun"),
    ("誤りを含む文 2（語順ミス）", "language natural is processing fun"),
]

rows = []
for label, s in test_sents:
    rows.append({"ケース": label, "文": s, "Perplexity (低いほど自然)": round(bigram_perplexity(s), 2)})
pd.DataFrame(rows)

,ケース,文,Perplexity (低いほど自然)
0,自然な文 1,i like an apple,1.77
1,誤りを含む文 1（冠詞ミス）,i like a apple,6.33
2,自然な文 2,natural language processing is fun,1.65
3,誤りを含む文 2（語順ミス）,language natural is processing fun,172.83
